In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

PARAMS = ["pH", "DO", "BOD", "COD", "TDS", "SS", "EC"]
TARGET = "WQI"

def prepare(path, name):
    df = pd.read_csv(path)

    df[PARAMS] = df[PARAMS].fillna(df[PARAMS].median())
    imputer = KNNImputer(n_neighbors=5)
    df[PARAMS] = imputer.fit_transform(df[PARAMS])

    df = df.drop_duplicates().reset_index(drop=True)

    df[PARAMS] = MinMaxScaler().fit_transform(df[PARAMS])
    df[PARAMS] = StandardScaler().fit_transform(df[PARAMS])

    X_train, X_test, y_train, y_test = train_test_split(
        df[PARAMS], df[TARGET], test_size=0.20, random_state=42
    )
    X_train.to_csv(f"{name}_X_train.csv", index=False)
    X_test.to_csv(f"{name}_X_test.csv", index=False)
    y_train.to_csv(f"{name}_y_train.csv", index=False)
    y_test.to_csv(f"{name}_y_test.csv", index=False)
    print(name, X_train.shape, X_test.shape)

prepare("dhaka_rivers.csv", "dhaka")
prepare("sylhet_rivers.csv", "sylhet")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

models = {
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "Bagging": BaggingRegressor(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
}

def evaluate(region):
    X_train = pd.read_csv(f"{region}_X_train.csv")
    X_test = pd.read_csv(f"{region}_X_test.csv")
    y_train = pd.read_csv(f"{region}_y_train.csv").iloc[:, 0]
    y_test = pd.read_csv(f"{region}_y_test.csv").iloc[:, 0]

    rows = []
    for name, reg in models.items():
        reg.fit(X_train, y_train)
        pred = reg.predict(X_test)
        rows.append({
            "Model": name,
            "R2": r2_score(y_test, pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
            "MAE": mean_absolute_error(y_test, pred),
            "MAPE": mape(y_test.values, pred),
        })
    out = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
    out.to_csv(f"{region}_model_results.csv", index=False)
    return out

print(evaluate("dhaka"))
print(evaluate("sylhet"))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor

PARAMS = ["pH", "DO", "BOD", "COD", "TDS", "SS", "EC"]

def analyze(region):
    X_train = pd.read_csv(f"{region}_X_train.csv")
    X_test = pd.read_csv(f"{region}_X_test.csv")
    y_train = pd.read_csv(f"{region}_y_train.csv").iloc[:, 0]
    y_test = pd.read_csv(f"{region}_y_test.csv").iloc[:, 0]

    reg = GradientBoostingRegressor(random_state=42)
    reg.fit(X_train, y_train)
    pred = reg.predict(X_test)

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, pred, alpha=0.6)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    plt.plot(lims, lims, "r--")
    plt.xlabel("Actual WQI")
    plt.ylabel("Predicted WQI")
    plt.tight_layout()
    plt.savefig(f"{region}_actual_vs_predicted.png", dpi=150)

    corr = pd.concat([X_train, y_train.rename("WQI")], axis=1).corr()
    plt.figure(figsize=(8, 7))
    sns.heatmap(corr, annot=True, fmt=".2f")
    plt.tight_layout()
    plt.savefig(f"{region}_correlation.png", dpi=150)

analyze("dhaka")
analyze("sylhet")
